In [ ]:
import requests # Apache License 2.0
from requests.auth import HTTPBasicAuth

import uuid     # in python
import base64   # in python
import yaml     # MIT
import json


from app.utils import get_data_offer, offer2et, create_poc_ContractRequest_body
from app.utils import str_edc_catalog

In [ ]:
# --- variables ---
# Endpoint
url_edc_consumer_control_plane_base = "https://fx-edc3-controlplane.factory-x.catena-x.net/"

# Headers
header_control_plane = {
    "X-API-Key": "jZtfmOrwzIK2FdEwJlR4",
    "Content-Type": "application/json"
}

# Proxy configuration 
proxies = {
    "http": "http://proxy01.wittag.local:9400",
    "https": "http://proxy01.wittag.local:9400"
}
proxies = {} #(disable for WITTENSTEIN setup)


edc_provider_bpn = "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:01cd5cbd-3338-463f-99bc-3f210e69b3c8:fx-demo-04"
url_edc_provider_control_plane_base = "https://fx-edc4-controlplane.factory-x.catena-x.net"

In [ ]:
# Headers
header_control_plane = {
    "X-API-Key": "jZtfmOrwzIK2FdEwJlR4",
    "Content-Type": "application/json"
}


# Request body
data = {
    "@context": {
        "@vocab": "https://w3id.org/edc/v0.0.1/ns/",
        "odrl": "http://www.w3.org/ns/odrl/2/"
    },
    "@type": "CatalogRequest",
    "counterPartyId": "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:01cd5cbd-3338-463f-99bc-3f210e69b3c8:fx-demo-04",
    "counterPartyAddress": "https://fx-edc4-controlplane.factory-x.catena-x.net/api/v1/dsp",
    "protocol": "dataspace-protocol-http",
    "querySpec": {
        "@type": "QuerySpec",
        "offset": 0,
        "limit": 10,
        "filterExpression": []
    }
}

url = url_edc_consumer_control_plane_base + "management/v3/catalog/request"

# Send request through proxy
response = requests.post(
    url,
    headers=header_control_plane,
    data=json.dumps(data),
    proxies=proxies,
    verify=False   # Often needed behind corporate proxies; remove if not needed
)

# Output
print("Status:", response.status_code)
print("Response:")
try:
    print(json.dumps(response.json(), indent=2))
except:
    print(response.text)

# DTR
# -------------------------------------------------------------------------------------------------------------------

In [ ]:
object_of_agreement_dtr = 'faaast-dtr-trumpf-vm' # we 'magically' know this due to the push notification

In [ ]:
# Initiating a Contract Negotiation
## Creating a new Contract Negotiation
init_edr_body = {
    "@context": [
        "https://w3id.org/tractusx/auth/v1.0.0",
        "https://w3id.org/catenax/2025/9/policy/context.jsonld",
        "https://w3id.org/catenax/2025/9/policy/odrl.jsonld",
        "https://w3id.org/dspace/2025/1/context.jsonld",
        "https://w3id.org/edc/dspace/v0.0.1",
        {
            "fx-policy": "https://w3id.org/factoryx/policy/v1.0/"
        }
    ],
    "@type": "https://w3id.org/edc/v0.0.1/ns/ContractRequest",
    "https://w3id.org/edc/v0.0.1/ns/counterPartyAddress": "https://fx-edc4-controlplane.factory-x.catena-x.net/api/v1/dsp/2025-1",
    "https://w3id.org/edc/v0.0.1/ns/protocol": "dataspace-protocol-http:2025-1",
    "https://w3id.org/edc/v0.0.1/ns/policy": {
        "@id": "V2l0dGVuc3RlaW5fQ29udHJhY3RfRGVmaW5pdGlvbl9hMDAy:ZmFhYXN0LWR0ci10cnVtcGYtdm0=:YTNhNTZiZWUtZjMyMS00OWY1LWFmMzUtNmNmMTczZGMyYTA5",
        "@type": "odrl:Offer",
        "odrl:assigner": {
            "@id": "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:01cd5cbd-3338-463f-99bc-3f210e69b3c8:fx-demo-04"
        },
        "odrl:permission": {
            "action": "odrl:use",
            "constraint": [
                {
                    "leftOperand": "fx-policy:BusinessPartnerDID",
                    "operator": "eq",
                    "rightOperand": "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:0d42e03e-2e0f-4901-a7f2-f7f9a2baa87e:fx-demo-03"
                }
            ]
        },
        "odrl:prohibition": [],
        "odrl:obligation": [],
        "odrl:target": {
            "@id": object_of_agreement_dtr
        }
    }
}

result = requests.post(url=url_edc_consumer_control_plane_base + 'management/v3/edrs', 
                                      headers=header_control_plane, json=init_edr_body,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)

ContractID = result.json()["@id"]
print("ContractID:", ContractID)

In [ ]:
# Checking for Completion

check_state_body = {}
url = url_edc_consumer_control_plane_base + f"management/v3/contractnegotiations/{ContractID}"
print(url)
result = requests.get(url=url, 
                                      headers=header_control_plane, json=check_state_body,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)
    

In [ ]:
#request edr entries and get transferProcessId
body = {
    "@context": {
        "@vocab": "https://w3id.org/edc/v0.0.1/ns/"
    },
    "@type": "QuerySpec",
    "filterExpression": [
        {
            "operandLeft": "contractNegotiationId",
            "operator": "=",
            "operandRight": ContractID
        }
    ]
}

url = url_edc_consumer_control_plane_base + f"management/v3/edrs/request"
print(url)
result = requests.post( url=url, 
                        headers=header_control_plane, json=body,
                        proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")

print(type(result.json()))
try:
    print(json.dumps(result.json(), indent=2))
except:
    print("NO JSON RESPONSE:")
    print(result.text)

transferProcessId = result.json()[0]["transferProcessId"]
print("transferProcessId:", transferProcessId)

In [ ]:

# get token

url = url_edc_consumer_control_plane_base + f"management/v3/edrs/{transferProcessId}/dataaddress?auto_refresh=true"
print(url)
result = requests.get(  url=url, 
                        headers=header_control_plane, 
                        proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
myJson = {}
try:
    myjson = result.json()
    print(json.dumps(myjson, indent=2))
except:
    print("NO JSON RESPONSE:")
    print(result.text)

authorization = myjson["authorization"]
authorization

In [ ]:

assetID_as_string = "https://wgrp.biz/aas/xdwayPN"

# Convert string to bytes
string_bytes = assetID_as_string.encode('utf-8')

# Encode to Base64
base64_bytes = base64.b64encode(string_bytes)

# Convert bytes back to string
assetID_base64 = base64_bytes.decode('utf-8')

print("Base64 Encoded:", assetID_base64)

In [ ]:
# get shelldescriptor

endpoint = "https://fx-edc4-dataplane.factory-x.catena-x.net/api/public"

# Headers
header = {
        "authorization": authorization,
}

url = endpoint + "/shell-descriptors/" + assetID_base64
print(url)
result = requests.get(url=url, 
                      headers=header,
                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)
    
# hard coded for now: should look for interface
endpointNumber = 1
endpoint_href = result.json()["endpoints"][endpointNumber]["protocolInformation"]["href"]
endpoint_href

 # AAS

In [ ]:
object_of_agreement_AASServer = 'faaast-service-trumpf-maas' # we 'magically' know this due to the push notification


In [ ]:
# Initiating a Contract Negotiation
## Creating a new Contract Negotiation
init_edr_body = {
    "@context": [
        "https://w3id.org/tractusx/auth/v1.0.0",
        "https://w3id.org/catenax/2025/9/policy/context.jsonld",
        "https://w3id.org/catenax/2025/9/policy/odrl.jsonld",
        "https://w3id.org/dspace/2025/1/context.jsonld",
        "https://w3id.org/edc/dspace/v0.0.1",
        {
            "fx-policy": "https://w3id.org/factoryx/policy/v1.0/"
        }
    ],
    "@type": "https://w3id.org/edc/v0.0.1/ns/ContractRequest",
    "https://w3id.org/edc/v0.0.1/ns/counterPartyAddress": "https://fx-edc4-controlplane.factory-x.catena-x.net/api/v1/dsp/2025-1",
    "https://w3id.org/edc/v0.0.1/ns/protocol": "dataspace-protocol-http:2025-1",
    "https://w3id.org/edc/v0.0.1/ns/policy": {
        "@id": "V2l0dGVuc3RlaW5fQ29udHJhY3RfRGVmaW5pdGlvbl9hMDAy:ZmFhYXN0LWR0ci10cnVtcGYtdm0=:YTNhNTZiZWUtZjMyMS00OWY1LWFmMzUtNmNmMTczZGMyYTA5",
        "@type": "odrl:Offer",
        "odrl:assigner": {
            "@id": "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:01cd5cbd-3338-463f-99bc-3f210e69b3c8:fx-demo-04"
        },
        "odrl:permission": {
            "action": "odrl:use",
            "constraint": [
                {
                    "leftOperand": "fx-policy:BusinessPartnerDID",
                    "operator": "eq",
                    "rightOperand": "did:web:dim-static-prod.dis-cloud-prod.cfapps.eu10-004.hana.ondemand.com:dim-hosted:0d42e03e-2e0f-4901-a7f2-f7f9a2baa87e:fx-demo-03"
                }
            ]
        },
        "odrl:prohibition": [],
        "odrl:obligation": [],
        "odrl:target": {
            "@id": object_of_agreement_AASServer
        }
    }
}

result = requests.post(url=url_edc_consumer_control_plane_base + 'management/v3/edrs', 
                                      headers=header_control_plane, json=init_edr_body,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)
    
ContractID = result.json()["@id"]
print("ContractID:", ContractID)
    

In [ ]:
# Checking for Completion

check_state_body = {}
url = url_edc_consumer_control_plane_base + f"management/v3/contractnegotiations/{ContractID}"
print(url)
result = requests.get(url=url, 
                                      headers=header_control_plane, json=check_state_body,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)
    

In [ ]:
#request edr entries and get transferProcessId
body = {
    "@context": {
        "@vocab": "https://w3id.org/edc/v0.0.1/ns/"
    },
    "@type": "QuerySpec",
    "filterExpression": [
        {
            "operandLeft": "contractNegotiationId",
            "operator": "=",
            "operandRight": ContractID
        }
    ]
}

url = url_edc_consumer_control_plane_base + f"management/v3/edrs/request"
print(url)
result = requests.post(url=url, 
                                      headers=header_control_plane, json=body,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)
    

transferProcessId = result.json()[0]["transferProcessId"]
print("transferProcessId:", transferProcessId)

In [ ]:
# get token


url = url_edc_consumer_control_plane_base + f"management/v3/edrs/{transferProcessId}/dataaddress?auto_refresh=true"
print(url)
result = requests.get(url=url, 
                                      headers=header_control_plane, 
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
json = {}
try:
    json = result.json()
    print(json.dumps(json, indent=2))
except:
    print("NO JSON RESPONSE:")
    print(result.text)

authorization = json["authorization"]
print("authorization:", authorization)

In [ ]:
#authorization = "eyJraWQiOiJ0eC1lZGMtZGFwcy1jZXJ0IiwiYWxnIjoiUlMyNTYifQ.eyJpc3MiOiJkaWQ6d2ViOmRpbS1zdGF0aWMtcHJvZC5kaXMtY2xvdWQtcHJvZC5jZmFwcHMuZXUxMC0wMDQuaGFuYS5vbmRlbWFuZC5jb206ZGltLWhvc3RlZDowMWNkNWNiZC0zMzM4LTQ2M2YtOTliYy0zZjIxMGU2OWIzYzg6ZngtZGVtby0wNCIsImF1ZCI6ImRpZDp3ZWI6ZGltLXN0YXRpYy1wcm9kLmRpcy1jbG91ZC1wcm9kLmNmYXBwcy5ldTEwLTAwNC5oYW5hLm9uZGVtYW5kLmNvbTpkaW0taG9zdGVkOjBkNDJlMDNlLTJlMGYtNDkwMS1hN2YyLWY3ZjlhMmJhYTg3ZTpmeC1kZW1vLTAzIiwic3ViIjoiZGlkOndlYjpkaW0tc3RhdGljLXByb2QuZGlzLWNsb3VkLXByb2QuY2ZhcHBzLmV1MTAtMDA0LmhhbmEub25kZW1hbmQuY29tOmRpbS1ob3N0ZWQ6MDFjZDVjYmQtMzMzOC00NjNmLTk5YmMtM2YyMTBlNjliM2M4OmZ4LWRlbW8tMDQiLCJleHAiOjE3Njk1OTEwNTksImlhdCI6MTc2OTU5MDc1OSwianRpIjoiM2M5NGUzYmMtMTU1OC00NjEwLWIxZDQtZGVkMjAyYTU5YjMzIn0.kot_PPlHxKVpV9nwU5_dt3KZ3_TVQk3XvtAKGG16LTblAERXot3kVFPfCetgjucSngvHm3zxWcmxict4Y8ybZQtsqcK6jrxx2d8GgScwB0DCswwizzo1N6hnMpJMRRbxFeOvOKySvGi6eaRw-gqwYkzN3qZFjvwEN9uqbqyICG9jL5zbiKipschRKkPRNbIqvyS0fH8GOZV9_by-OaJ5KLKH4TBZdVfxQ5u6ICS4Ru5Nj5P_QUX80wpu89tHV7Cpz_LBAU4gRrcPXu0rEqEbb2YA-J-yWk0DBBj4Nwo47hsqchTTz98LWdgA_FJ6sop73BaKCslakl9FbXTeo7FjVB"

In [ ]:
# get shell

endpoint = endpoint_href
# https://fx-edc4-dataplane.factory-x.catena-x.net/api/public/api/v3.0/shells/VHJ1TGFzZXJfNTA2MF9maWJlcg==
# Headers
header = {
        "authorization": authorization,
}

url = endpoint
print(url)
result = requests.get(url=url, 
                        headers=header,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)

In [ ]:
# get shell (TEST with hardcoded endpoint)

endpoint = "https://fx-edc4-dataplane.factory-x.catena-x.net/api/public/v3.0/shells/aHR0cHM6Ly93Z3JwLmJpei9hYXMveGR3YXlQTg=="
# 
# Headers
header = {
        "authorization": authorization,
}

url = endpoint
print(url)
result = requests.get(url=url, 
                                      headers=header,
                                      proxies=proxies, verify=False)
# Output
print("Status:", result.status_code)
print("Response:")
try:
    print(json.dumps(result.json(), indent=2))
except:
    print(result.text)